[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/gravitational_waves/rendering.ipynb)

# Inside a Closed Universe

This example renders a closed universe, the round three-sphere, from inside, by walking each ray of light through a mesh of tetrahedra that fills it. Light in the three-sphere runs along great circles, so each edge of a mesh, an arc of a great circle, is seen as a straight strut, and struts fill the view in every direction; past a quarter turn the great circles from the eye close in again on its antipode, and far struts loom larger. A ray crosses one tetrahedron at a time: inside a tetrahedron it runs straight, and it leaves through a face, as through a portal, into the neighbour across it.

Points of the three-sphere are modelled as antivectors of unit size in the algebra of four directions, the origin `mv.w.dual()` the pole of its equator `mv.w`. Every point carries an frame, the motor `frame(place)` that carries the origin to it by turning alike in two square planes. The three-sphere has such frames smoothly everywhere at once, and in them a ray's heading keeps still: from a place, a ray rolls along its great circle by `frame(place) * (moving(heading) * (-length / 2)).exp()`, where `moving` is the map from a step of the origin to the rate that moves the origin along it. A tetrahedron's ramps, the planes through its faces scaled to read one at the opposite corners, read a point's barycentric weights, `ramps & point`. Their sum is the tetrahedron's chart, the plane through its corners, onto which its piece of the sphere is projected from the centre: on it every great circle runs straight, and the weights change evenly along it.

In the notation of Lie groups, the unit three-sphere is the group $\mathrm{SU}(2)$ with its bi-invariant metric; its left-invariant frame field is $e_i(q) = q\,\tau_i$, and the geodesics from $q$ are the curves $q\exp(t\,\xi)$ for a fixed $\xi \in \mathfrak{su}(2)$, whose velocity $q \exp(t\,\xi)\,\xi$ has the same components $\xi$ in that frame all along.

`examples.relativity.gravitational_waves.web.page()` also writes a page that flies through the 600-cell as the slowest wave rings, traced on the reader's own GPU in the browser.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from functools import reduce
from operator import and_

import numpy as np
from numga import Algebra, NumpyContext

from examples.util.animation import show_animation
from examples.relativity.gravitational_waves import render, scenarios
from examples.relativity.gravitational_waves.core import ga

mv = NumpyContext(ga).multivector
# Planes, and those through the origin, the polar planes of its steps.
Plane = ga.gatype.vector()                                                      # Vector+
Through = Plane.derive.within(Algebra("x+y+z+"))                                # Vector+
# Points of the sphere and the origin's steps; the motors that carry them.
Point = ga.gatype.antivector()                                                  # Antivector-
Step = Through.derive.dual                                                      # Antivector-
Motor = ga.gatype.rotor()                                                       # Rotor+
Translating = ga.gatype((Step, Through))                                        # Antivector- <- Vector+

# The origin's equator, the plane `w`, and the origin, its pole.
equator = mv.w                                                                  # [] Plane+
origin = equator.dual()                                                         # [] Point-
# The step the origin takes under the motion across a plane through it, a step by its value, cast so; its inverse
# gives each step its polar plane, and the meet of the equator with that plane is the rate that moves the origin along
# the step.
polar = origin.commutator(equator ^ Through).cast(Translating).inverse()        # [] Plane+ <- Point-
moving = equator ^ polar(Step)                                                  # [] Rate+ <- Point-

## The frame at each place

In four dimensions the product of a rate with the extrinsic pseudoscalar `mv.xyzw` is the rate in the plane square to it, and as the pseudoscalar squares to one, it splits every rate into a half it keeps and a half it flips. Each half turns alike in two square planes, by the same angle in both, so that it moves every point of the sphere the same distance along a great circle. The quotient `place / origin` is the turn that carries the origin to a place in the plane of the great circle between them. Its half that the pseudoscalar keeps, with one in the other half, still carries the origin there, turning alike in that plane and in the plane square to it: that motor is the frame at the place. The pseudoscalar is what picks out the turns alike in two square planes, and with them a frame at every place that varies smoothly over the whole sphere.

In [ ]:
# The extrinsic pseudoscalar, whose product with a rate is the rate in the plane square to it.
pseudoscalar = mv.xyzw                                                          # [] Antiscalar+


def frame(places: Point) -> Motor:
    """The frames at places on the sphere, `Motor+`: the motors that carry the origin to each by turning
    alike in the plane of the great circle between them and in the plane square to it."""
    # The half of the quotient of the place by the origin that the pseudoscalar keeps, with one in the other half.
    return ((1 + pseudoscalar) * (places.normalized() / origin) + (1 - pseudoscalar)) / 2

## Legs across the tetrahedra

On a tetrahedron's chart a ray's great circle is straight and the barycentric weights change evenly along it, so one straight leg takes the ray across the whole tetrahedron, to the face whose corner's weight runs out first. The topology names the tetrahedron across each face, and since a heading keeps still in the frames, the next leg starts there with the same heading. The walk is the part of the GPU's shader that knows the sphere: each sample's ray takes legs from the eye until it meets a strut about an edge or has run half a turn, and the renderer draws what it meets.

In [ ]:
# A leg's reach, past any tetrahedron's far face and short of its chart's horizon, a quarter turn from its middle.
reach = 0.6
# A ray's leg across a tetrahedron, in numga's dialect: from a place along a heading, a unit step of the origin in the
# frame there, having entered by a face, four where it starts inside, at a time that the round sphere leaves alone.
walk = """
// A tetrahedron of the mesh: across the face facing each corner the neighbour and the face it is entered by there,
// and the ramp of each corner.
struct Tet {
    neighbours: vec4<u32>,
    entries: vec4<u32>,
    ramps: array<Plane, 4>,
}

var<storage, read> tets: array<Tet>;
var<uniform> origin: Point;
var<uniform> moving: Moving;
var<uniform> reach: f32;

// The chart's plane, where the leg starts on it, the straight segment it runs, the arc it covers, the heading at its
// end, and the tetrahedron and face the next leg starts in.
struct Leg {
    plane: Plane,
    chart: Point,
    segment: Point,
    length: f32,
    heading: Step,
    tet: u32,
    entry: u32,
}

fn frame(place: Point) -> Motor {
    return ((1.0 + mv.xyzw) * (place / origin) + (1.0 - mv.xyzw)) * 0.5;
}

fn next_leg(place: Point, heading: Step, tet: u32, entry: u32, time: f32) -> Leg {
    // The chart, the plane that reads one at each corner; the place on it, and the straight line on it to where the
    // ray's great circle is a reach on.
    let unit = tets[tet].ramps[0] + tets[tet].ramps[1] + tets[tet].ramps[2] + tets[tet].ramps[3];
    let chart = place / (unit & place);
    let reach_end = (frame(place) * (moving(heading) * (-0.5 * reach)).exp()) >> origin;
    let segment = reach_end / (unit & reach_end) - chart;
    // The share of the line where the first corner's weight runs out; a place a hair outside the face it is on, by
    // round-off, leaves at once.
    var share = 1.0;
    var exit = 4u;
    for (var corner = 0u; corner < 4u; corner++) {
        let change = tets[tet].ramps[corner] & segment;
        if (corner != entry && change < 0.0) {
            let run_out = max(-(tets[tet].ramps[corner] & chart) / change, 0.0);
            if (run_out < share) {
                share = run_out;
                exit = corner;
            }
        }
    }
    let end = (chart + segment * share).normalized();
    var this_leg = Leg(unit, chart, segment * share, acos(clamp(place.dot(end), -1.0, 1.0)), heading, tet, 4u);
    if (exit < 4u) {
        this_leg.tet = tets[tet].neighbours[exit];
        this_leg.entry = tets[tet].entries[exit];
    }
    return this_leg;
}
"""

## A flight through the 600-cell

The 600-cell's 120 vertices lie on the sphere, the turns their frames make carry them onto themselves, and round, without waves, each of its 600 regular tetrahedra is a piece of the sphere as it is. One of its Boerdijk–Coxeter helices, a column of thirty tetrahedra stacked face to face that closes on itself once round the sphere, gives the flight its path: the helix's axis threads a tunnel through its tetrahedra, clear of every edge. A camera is a motor that carries the origin to the eye and the origin's steps along `x`, `y` and `z` to the eye's right, up and forward; on the axis, it looks along it, and the turn that moves the origin along `z`, after the camera, flies it ahead, back where it started once round the sphere. Drawing the flight takes about five seconds, most of it tracing the frames.

In [ ]:
# The frames once round the sphere, and a frame's duration in milliseconds.
n_flight_frames = 90
duration_ms = 40

In [ ]:
# The 600-cell; one of its Boerdijk–Coxeter helices, its rails, a point on its axis, and there the axis's direction and
# a step across it, steps of the origin in the frame at the point; and the great circles its edges run along.
cell_mesh, cell_vertices = scenarios.six_hundred_cell()
rails, axis_point, axis_direction, side_direction = scenarios.helix(cell_vertices)
circles = scenarios.great_circles(cell_mesh, cell_vertices)
# Each corner's ramp: the plane through the face across from it, the join of that face's corners, scaled to read one
# at the corner.
cell_corners = cell_mesh.incidence[3, 0] * cell_vertices                        # [corners] Point-[tets]
cell_faces = reduce(and_, cell_corners[cell_mesh.corners[3, 2]])                # [corners] Plane-[tets]
cell_ramps = cell_faces / (cell_faces & cell_corners)                           # [corners] Plane+[tets]


def facing(placing: Motor, forward: Point, up: Point) -> Motor:
    """A camera: the motor `placing`, which carries the origin to the eye, after the turns that carry the origin's step
    along z to `forward` and then, in the plane square to it, the one along y to `up`, unit steps of the origin square
    to each other in the eye's frame."""
    # The turn that carries one unit step to another in their plane is the square root of their quotient.
    aiming = (forward / mv.z.dual()).square_root()                              # [] Motor+
    rolling = (up / (aiming >> mv.y.dual())).square_root()                      # [] Motor+
    return placing * rolling * aiming


# The camera on the axis, looking along it with the step across it up, flown ahead along its view once round.
flight_start = facing(frame(axis_point), axis_direction, side_direction)        # [] Motor+
forward_rate = moving(mv.z.dual())                                              # [] Rate+
distances = np.linspace(0, 2 * np.pi, n_flight_frames, endpoint=False)          # [frames]
flight = flight_start * (forward_rate * (-distances / 2)).exp()                 # [frames] Motor+
cell_walk_values = dict(ramps=cell_ramps, origin=origin, moving=moving, reach=reach)
show_animation(render.polygon_view(walk, cell_walk_values, cell_mesh, cell_vertices, flight, circles),
               "gravitational_waves_flight", duration_ms)

Each of the 600-cell's 72 great circles of edges, ten edges to each, is drawn in its own colour, so that a strut's colour carries on through the vertices along its circle; past a quarter turn the struts come back in, larger, as the circles close in on the antipode.

## The helix seen from outside

Seen from outside, the helix's thirty tetrahedra wind its three rails about the axis, linked by rungs. The camera stands most of a quarter turn out from the axis and looks back at it, the axis up, rolled in its own plane `xy` so that the axis leans. The turn in the axis's own plane carries the camera along the axis while it keeps every point of the square plane still, so the camera passes the helix without turning about it, and the rails corkscrew about the axis.

In [ ]:
# The frames once round the axis; how far out from the axis the camera stands, and how far it is rolled.
n_pan_frames = 60
pan_distance = 1.02
pan_roll = 0.36

In [ ]:
# The camera that far out across the axis, looking back at it with the axis up, rolled in its own plane `xy`.
station = frame(axis_point) * (moving(side_direction) * (-pan_distance / 2)).exp()       # [] Motor+
pan_start = facing(station, -side_direction, axis_direction) * (mv.xy * (pan_roll / 2)).exp()   # [] Motor+
# The turn along the axis: the rate that moves the point on it along its direction, carried there.
axis_rate = frame(axis_point) >> moving(axis_direction)                         # [] Rate+
angles = np.linspace(0, 2 * np.pi, n_pan_frames, endpoint=False)                # [frames]
pan = (axis_rate * (-angles / 2)).exp() * pan_start                             # [frames] Motor+
show_animation(render.helix_view(walk, cell_walk_values, cell_mesh, cell_vertices, pan, rails),
               "gravitational_waves_helix", duration_ms)

The helix's three rails are drawn each in its own colour, the rungs between them light, and every other edge a thin ghost.

## A flight through an irregular mesh

The same walk runs through any mesh of the sphere. This one holds the 600-cell's vertices and a few orbits of random points under the turns that carry the 600-cell onto itself, relaxed until its tetrahedra are as round as the orbits allow. Its edges follow no shared great circles and are drawn plain, and a straight path through so many edges passes through some of them: the renderer leaves out a strut while the eye is inside it. This cell takes about ten seconds, most of it relaxing the mesh and tracing the frames.

In [ ]:
irregular_mesh, irregular_vertices = scenarios.irregular_three_sphere()
irregular_corners = irregular_mesh.incidence[3, 0] * irregular_vertices         # [corners] Point-[tets]
irregular_faces = reduce(and_, irregular_corners[irregular_mesh.corners[3, 2]])   # [corners] Plane-[tets]
irregular_ramps = irregular_faces / (irregular_faces & irregular_corners)       # [corners] Plane+[tets]
# From the middle of its first tetrahedron, looking along the origin's step along z there, flown ahead once round.
middle = (irregular_mesh.incidence[3, 0].mean(axis=0) * irregular_vertices).batch()[0].normalized()   # [] Point-
irregular_flight = frame(middle) * (forward_rate * (-distances / 2)).exp()      # [frames] Motor+
show_animation(render.plain_view(walk, dict(ramps=irregular_ramps, origin=origin, moving=moving, reach=reach),
                                 irregular_mesh, irregular_vertices, irregular_flight),
               "gravitational_waves_irregular", duration_ms)

In [ ]:
# checks
# The frames carry the origin to each place; flown once round, the camera is back where it started; and the
# helix is thirty tetrahedra, every one with all four corners on its rails.
misses = (frame(cell_vertices) >> origin) - cell_vertices                       # Point-[vertices]
assert (misses.dot(misses).square_root() < 1e-13).all()
once_round = (flight_start * (forward_rate * -np.pi).exp()) >> origin           # [] Point-
back = once_round - (flight_start >> origin)                                    # [] Point-
assert back.dot(back).square_root() < 1e-13
assert np.isin(cell_mesh.elements[3], rails).all(axis=-1).sum() == 30